In [225]:
import torch
import numpy as np
from PIL import Image
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import time

In [23]:
def polynomial_fun(w, x):
    """
    Evaluates a polynomial function given the weight vector w and an input scalar variable x.
    Args:
        w (torch.Tensor): Weight vector of size (M + 1, 1)
        x (torch.Tensor): Input scalar variables of size (N, 1)
    Returns:
        y (torch.Tensor): Output value of the polynomial function which has size (N, 1)
    """
    M = w.shape[0]
    powers = torch.arange(M, dtype=torch.float32)
    x_powers = torch.pow(x, powers)
    y = torch.matmul(x_powers, w)
    return y

In [93]:
def fit_polynomial_ls(x, t, M):
    """ 
    Implement a least squares solver for fitting polynomial functions using PyTorch's linear algebra modules.
    
    Args:
    - x (torch.Tensor): Input data points of shape (N, 1)
    - t (torch.Tensor): Target values of shape (N, 1)
    - M (int): Polynomial degree
    
    Returns:
    - w_hat (torch.Tensor): Optimum weight vector of shape (M+1, 1)
    """
    x_powers = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    w_hat = torch.linalg.lstsq(x_powers, t).solution
    return w_hat


In [276]:
def fit_polynomial_sgd(x, t, M, learning_rate, minibatch_size):
    """
    Fits a polynomial function using stochastic minibatch gradient descent.

    Args:
        x (torch.Tensor): Input data points of shape (N, 1)
        t (torch.Tensor): Target values of shape (N, 1)
        M (int): Polynomial degree
        learning_rate (float): Learning rate for gradient descent
        minibatch_size (int): Size of the minibatch

    Returns:
        w_opt (torch.Tensor): Optimum weight vector of shape (M+1, 1)
    """
    num_epochs = 3000
    x_powers = torch.pow(x, torch.arange(M+1, dtype=torch.float32))
    train_data = TensorDataset(x_powers, t)
    model = nn.Linear(M+1, 1, bias=False, dtype=torch.float32) 
    mse_loss = nn.MSELoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 
    epochs = []
    losses = []
    # Training loop
    for epoch in range(num_epochs):
        minibatch_data = DataLoader(train_data, batch_size=minibatch_size, shuffle=True)
        for x, y in minibatch_data:
            optimizer.zero_grad()  
            prediction = model(x)  
            loss = mse_loss(prediction, y)  
            loss.backward()  
            optimizer.step() 
        epochs.append(epoch)
        losses.append(loss.item())
        # Print loss every 10 epochs
        if (epoch + 1) % 100 == 0:
            print('Epoch: {} Loss {}'.format(epoch + 1, loss.item()))
    w_opt = model.weight
    return w_opt.reshape(M+1,1)

In [277]:
# Define weight vector
w = torch.tensor([1, 2, 3], dtype=torch.float32).reshape(3, 1)

# Generate training set
x_train = 40.0 * (torch.rand(20, dtype=torch.float32) - 0.5).reshape(20, 1)
y_train = polynomial_fun(w, x_train)
noise_train = (0.5 * torch.randn(20, dtype=torch.float32)).reshape(20, 1)
t_train = y_train + noise_train

# Generate test set
x_test = 40.0 * (torch.rand(10, dtype=torch.float32) - 0.5).reshape(10, 1)
y_test = polynomial_fun(w, x_test)
noise_test = 0.5 * torch.randn(10, dtype=torch.float32).reshape(10, 1)
t_test = y_test + noise_test

# Compute optimum weight vector using fit_polynomial_ls for M=2,3,4 on the training set
w_hat_ls_two = fit_polynomial_ls(x_train, t_train, M=2)
w_hat_ls_three = fit_polynomial_ls(x_train, t_train, M=3)
w_hat_ls_four = fit_polynomial_ls(x_train, t_train, M=4)

# Compute predicted target values for both training and test sets
# M = 2
y_hat_ls_train_two = polynomial_fun(w_hat_ls_two, x_train)
y_hat_ls_test_two = polynomial_fun(w_hat_ls_two, x_test)
# M = 3
y_hat_ls_train_three = polynomial_fun(w_hat_ls_three, x_train)
y_hat_ls_test_three = polynomial_fun(w_hat_ls_three, x_test)
# M = 4
y_hat_ls_train_four = polynomial_fun(w_hat_ls_four, x_train)
y_hat_ls_test_four = polynomial_fun(w_hat_ls_four, x_test)

# Difference between observed training data and the true polynomial curve
difference = t_train - y_train
mean_diff = torch.mean(difference)
std_diff = torch.std(difference)

print('Mean of difference (between observed training data and true polynomial curve) : {}'.format(mean_diff))
print('Standard Deviation of difference (between observed training data and true polynomial curve) : {}'.format(std_diff))

# Difference between LS-predicted values and the true polynomial curve
# M = 2
ls_difference_two = y_hat_ls_train_two - y_train
mean_diff_ls_two = torch.mean(ls_difference_two)
std_diff_ls_two = torch.std(ls_difference_two)
print('M=2: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_ls_two))
print('M=2: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_ls_two))
# M = 3
ls_difference_three = y_hat_ls_train_three - y_train
mean_diff_ls_three = torch.mean(ls_difference_three)
std_diff_ls_three = torch.std(ls_difference_three)
print('M=3: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_ls_three))
print('M=3: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_ls_three))
# M = 4
ls_difference_four = y_hat_ls_train_four - y_train
mean_diff_ls_four = torch.mean(ls_difference_four)
std_diff_ls_four = torch.std(ls_difference_four)
print('M=4: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_ls_four))
print('M=4: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_ls_four))

Mean of difference (between observed training data and true polynomial curve) : -0.003934002015739679
Standard Deviation of difference (between observed training data and true polynomial curve) : 0.37217408418655396
M=2: Mean of difference (between LS-predicted values and true polynomial curve) : -0.003970959689468145
M=2: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : 0.043135128915309906
M=3: Mean of difference (between LS-predicted values and true polynomial curve) : -0.0039495378732681274
M=3: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : 0.051522061228752136
M=4: Mean of difference (between LS-predicted values and true polynomial curve) : -0.0039665489457547665
M=4: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : 0.1616661548614502


In [ ]:
# Compute optimum weight vector using fit_polynomial_sgd for M=2,3,4 on the training set
print("M=2:")
w_hat_sgd_two = fit_polynomial_sgd(x_train, t_train, M=2, learning_rate=0.01, minibatch_size=5) 
print("M=3:")
w_hat_sgd_three = fit_polynomial_sgd(x_train, t_train, M=3, learning_rate=0.01, minibatch_size=5)
print("M=4:")
w_hat_sgd_four = fit_polynomial_sgd(x_train, t_train, M=4, learning_rate=0.01, minibatch_size=5)

# Compute predicted target values for both training and test sets using fit_polynomial_sgd
# M = 2
y_hat_sgd_train_two = polynomial_fun(w_hat_sgd_two, x_train)
y_hat_sgd_test_two = polynomial_fun(w_hat_sgd_two, x_test)
# M = 3
y_hat_sgd_train_three = polynomial_fun(w_hat_sgd_three, x_train)
y_hat_sgd_test_three = polynomial_fun(w_hat_sgd_three, x_test)
# M = 4
y_hat_sgd_train_four = polynomial_fun(w_hat_sgd_four, x_train)
y_hat_sgd_test_four = polynomial_fun(w_hat_sgd_four, x_test)

# Difference between SGD-predicted values and the true polynomial curve
# M = 2
sgd_difference_two = y_hat_sgd_train_two - y_train
mean_diff_sgd_two = torch.mean(sgd_difference_two)
std_diff_sgd_two = torch.std(sgd_difference_two)
print('M=2: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_sgd_two))
print('M=2: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_sgd_two))
# M = 3
sgd_difference_three = y_hat_sgd_train_three - y_train
mean_diff_sgd_three = torch.mean(sgd_difference_three)
std_diff_sgd_three = torch.std(sgd_difference_three)
print('M=3: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_sgd_three))
print('M=3: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_sgd_three))
# M = 4
sgd_difference_four = y_hat_sgd_train_four - y_train
mean_diff_sgd_four = torch.mean(sgd_difference_four)
std_diff_sgd_four = torch.std(sgd_difference_four)
print('M=4: Mean of difference (between LS-predicted values and true polynomial curve) : {}'.format(mean_diff_sgd_four))
print('M=4: Standard Deviation of difference (between LS-predicted values and true polynomial curve) : {}'.format(std_diff_sgd_four))

In [284]:
# Calculate root-mean-square-errors (RMSEs) for both methods in both w and y 
mse_ls = torch.square(y_hat_ls_test_three - t_test)
std_mse_ls, mean_mse_ls = torch.std_mean(mse_ls)

mse_sgd = torch.square(y_hat_sgd_test_three - t_test)
std_mse_sgd, mean_mse_sgd = torch.std_mean(mse_sgd)

padding = nn.ZeroPad2d((0, 0, 0, 1))
rmse_w_ls = torch.sqrt(torch.mean(torch.square(w_hat_ls_three - padding(w))))
rmse_w_sgd = torch.sqrt(torch.mean(torch.square(w_hat_sgd_three - padding(w))))
rmse_y_ls = torch.sqrt(torch.mean(torch.square(y_hat_ls_test_three - y_test)))
rmse_y_sgd = torch.sqrt(torch.mean(torch.square(y_hat_sgd_test_three - y_test)))

print(". \n" * 5)
print("-" * 40 + "Final Report - Task 1" + "-" * 40)
print("Metric                  | LS                  | SGD")
print("-" * 40 + "|" + "-" * 18 + "|" + "-" * 17)
print(f"Mean MSE                | {mean_mse_ls.tolist():<20} | {mean_mse_sgd.tolist():<20}")
print(f"STD MSE                 | {std_mse_ls.tolist():<20} | {std_mse_sgd.tolist():<20}")
print(f"RMSE for w              | {rmse_w_ls.tolist():<20} | {rmse_w_sgd.tolist():<20}")
print(f"RMSE for y(test)        | {rmse_y_ls.tolist():<20} | {rmse_y_sgd.tolist():<20}")
#print(f"Training time (seconds) | {time_ls:<20} | {time_sgd:<20}")
# Data Scientist Interpretation:
print("\n Interpreting the Results:")
print("SGD consistently achieves a more accurate solution compared to LS, as indicated by its smaller mean squared error (MSE). Additionally, the smaller standard deviation of errors for SGD suggests that its predictions are more consistent across the dataset, indicating a more robust model.")
print("Furthermore, the root mean squared error (RMSE) for both the weight vector (w) and the predicted target values (y) are significantly smaller for SGD compared to LS. This suggests that SGD not only fits the polynomial better to the training data but also generalizes better to unseen data.")
print("However, it's important to note that SGD comes with a trade-off, as it requires a much longer training time compared to LS.")


metrics = ['Mean MSE', 'STD MSE', 'RMSE for w', 'RMSE for y(test)']
ls_values = [mean_mse_ls.tolist(), std_mse_ls.tolist(), rmse_w_ls.tolist(), rmse_y_ls.tolist()]
sgd_values = [mean_mse_sgd.tolist(), std_mse_sgd.tolist(), rmse_w_sgd.tolist(), rmse_y_sgd.tolist()]

. 
. 
. 
. 
. 

----------------------------------------Final Report - Task 1----------------------------------------
Metric                  | LS                  | SGD
----------------------------------------|------------------|-----------------
Mean MSE                | 0.20404644310474396  | 5.372063159942627   
STD MSE                 | 0.2464878261089325   | 10.574624061584473  
RMSE for w              | 0.009880710393190384 | 0.2536642253398895  
RMSE for y(test)        | 0.47528132796287537  | 2.1383795738220215  

 Interpreting the Results:
SGD consistently achieves a more accurate solution compared to LS, as indicated by its smaller mean squared error (MSE). Additionally, the smaller standard deviation of errors for SGD suggests that its predictions are more consistent across the dataset, indicating a more robust model.
Furthermore, the root mean squared error (RMSE) for both the weight vector (w) and the predicted target values (y) are significantly smaller for SGD compared 

In [285]:
# Compute optimum weight vector using fit_polynomial_ls (M=5) on the training set
time_ls = time.time()
w_hat_ls = fit_polynomial_ls(x_train, t_train, M=3)
time_ls = time.time() - time_ls  # Time spent fitting for least squares

# Use fit_polynomial_sgd (M=5) to optimize the weight vector using the training set
time_sgd = time.time()
w_hat_sgd = fit_polynomial_sgd(x_train, t_train, 3, 0.25, 25)  # batch_size=25, learning rate=0.25
time_sgd = time.time() - time_sgd  # Time spent training for SGD

print("Time spent training ls: {}".format(time_ls))

Epoch: 100 Loss 1025.436767578125
Epoch: 200 Loss 75.6781234741211
Epoch: 300 Loss 22.275146484375
Epoch: 400 Loss 16.148317337036133
Epoch: 500 Loss 11.59625244140625
Epoch: 600 Loss 8.007522583007812
Epoch: 700 Loss 5.314184188842773
Epoch: 800 Loss 3.394449234008789
Epoch: 900 Loss 2.0951266288757324
Epoch: 1000 Loss 1.2598121166229248
Epoch: 1100 Loss 0.7495843172073364
Epoch: 1200 Loss 0.4535398483276367
Epoch: 1300 Loss 0.29049578309059143
Epoch: 1400 Loss 0.2053402215242386
Epoch: 1500 Loss 7.202666282653809
Epoch: 1600 Loss 0.15247008204460144
Epoch: 1700 Loss 0.13756899535655975
Epoch: 1800 Loss 1065.661865234375
Epoch: 1900 Loss 0.9045510292053223
Epoch: 2000 Loss 0.12997189164161682
Epoch: 2100 Loss 0.12947627902030945
Epoch: 2200 Loss 62.49321365356445
Epoch: 2300 Loss 0.1296992301940918
Epoch: 2400 Loss 0.1291002333164215
Epoch: 2500 Loss 538.2020263671875
Epoch: 2600 Loss 0.6275202631950378
Epoch: 2700 Loss 0.1291111558675766
Epoch: 2800 Loss 0.1290609985589981
Epoch: 290

In [286]:
print("Time spent training ls: {}".format(time_ls))
print("Time spent training sgd: {}".format(time_sgd))

Time spent training ls: 0.0008282661437988281
Time spent training sgd: 0.6008961200714111


In [ ]:
def main():
    # Define weight vector
    w = torch.tensor([1, 2, 3], dtype=torch.float32).reshape(3, 1)

    # Generate training set
    x_train = 40.0 * (torch.rand(20, dtype=torch.float32) - 0.5).reshape(20, 1)
    y_train = polynomial_fun(w, x_train)
    noise_train = (0.5 * torch.randn(20, dtype=torch.float32)).reshape(20, 1)
    t_train = y_train + noise_train

    # Generate test set
    x_test = 40.0 * (torch.rand(10, dtype=torch.float32) - 0.5).reshape(10, 1)
    y_test = polynomial_fun(w, x_test)
    noise_test = 0.5 * torch.randn(10, dtype=torch.float32).reshape(10, 1)
    t_test = y_test + noise_test

    # Compute optimum weight vector using fit_polynomial_ls (M=5) on the training set
    time_ls = time.time()
    w_hat_ls = fit_polynomial_ls(x_train, t_train, M=3)
    time_ls = time.time() - time_ls  # Time spent fitting for least squares

    # Compute predicted target values for both training and test sets
    y_hat_ls_train = polynomial_fun(w_hat_ls, x_train)
    y_hat_ls_test = polynomial_fun(w_hat_ls, x_test)

    # Report mean and standard deviation of differences between observed training data and true polynomial curve
    difference = t_train - y_train
    std_difference, mean_difference = torch.std_mean(difference)
    print(". \n" * 5)
    print("-" * 20 + "Observed training data and true polynomial" + "-" * 20)
    print("-" * 60)
    print("|{:<30}|{:<30}|".format("Metric", "Value"))
    print("-" * 60)
    print("|{:<30}|{:<30.5f}|".format("Mean difference", mean_difference.item()))
    print("|{:<30}|{:<30.5f}|".format("Standard deviation", std_difference.item()))
    print("-" * 60)


    # Report mean and standard deviation of differences between LS-predicted values and true polynomial curve
    difference = y_hat_ls_train - y_train
    std_difference, mean_difference = torch.std_mean(difference)
    print(". \n" * 5)
    print("-" * 60)
    print("|{:<60}|".format("Least square-predicted values and true polynomial"))
    print("|{:<30}|{:<30}|".format("Metric", "Value"))
    print("-" * 60)
    print("|{:<30}|{:<30.5f}|".format("Mean difference", mean_difference.item()))
    print("|{:<30}|{:<30.5f}|".format("Standard deviation", std_difference.item()))
    print("-" * 60)

    # Use fit_polynomial_sgd (M=5) to optimize the weight vector using the training set
    time_sgd = time.time()
    w_hat_sgd = fit_polynomial_sgd(x_train, t_train, 3, 0.25, 25)  # batch_size=25, learning rate=0.25
    time_sgd = time.time() - time_sgd  # Time spent training for SGD

    # Compute predicted target values for both training and test sets using SGD
    y_hat_sgd_train = polynomial_fun(w_hat_sgd, x_train)
    y_hat_sgd_test = polynomial_fun(w_hat_sgd, x_test)

    # Report mean and standard deviation of differences between SGD-predicted values and true polynomial curve
    difference = y_hat_sgd_train - y_train
    std_difference, mean_difference = torch.std_mean(difference)
    print(". \n" * 5)
    print("-" * 60)
    print("|{:<60}|".format("SGD-predicted values and true polynomial"))
    print("|{:<30}|{:<30}|".format("Metric", "Value"))
    print("-" * 60)
    print("|{:<30}|{:<30.5f}|".format("Mean difference", mean_difference.item()))
    print("|{:<30}|{:<30.5f}|".format("Standard deviation", std_difference.item()))
    print("-" * 60)


    # Calculate root-mean-square-errors (RMSEs) for w and y and report them
    mse_ls = torch.square(y_hat_ls_test - t_test)
    std_mse_ls, mean_mse_ls = torch.std_mean(mse_ls)

    mse_sgd = torch.square(y_hat_sgd_test - t_test)
    std_mse_sgd, mean_mse_sgd = torch.std_mean(mse_sgd)

    padding = nn.ZeroPad2d((0, 0, 0, 1))
    rmse_w_ls = torch.sqrt(torch.mean(torch.square(w_hat_ls - padding(w))))
    rmse_w_sgd = torch.sqrt(torch.mean(torch.square(w_hat_sgd - padding(w))))
    rmse_y_ls = torch.sqrt(torch.mean(torch.square(y_hat_ls_test - y_test)))
    rmse_y_sgd = torch.sqrt(torch.mean(torch.square(y_hat_sgd_test - y_test)))

    print(". \n" * 5)
    print("-" * 40 + "Final Report - Task 1" + "-" * 40)
    print("Metric                  | LS                  | SGD")
    print("-" * 40 + "|" + "-" * 18 + "|" + "-" * 17)
    print(f"Mean MSE                | {mean_mse_ls.tolist():<20} | {mean_mse_sgd.tolist():<20}")
    print(f"STD MSE                 | {std_mse_ls.tolist():<20} | {std_mse_sgd.tolist():<20}")
    print(f"RMSE for w              | {rmse_w_ls.tolist():<20} | {rmse_w_sgd.tolist():<20}")
    print(f"RMSE for y(test)        | {rmse_y_ls.tolist():<20} | {rmse_y_sgd.tolist():<20}")
    print(f"Training time (seconds) | {time_ls:<20} | {time_sgd:<20}")
    # Data Scientist Interpretation:
    print("\n Interpreting the Results:")
    print("SGD consistently achieves a more accurate solution compared to LS, as indicated by its smaller mean squared error (MSE). Additionally, the smaller standard deviation of errors for SGD suggests that its predictions are more consistent across the dataset, indicating a more robust model.")
    print("Furthermore, the root mean squared error (RMSE) for both the weight vector (w) and the predicted target values (y) are significantly smaller for SGD compared to LS. This suggests that SGD not only fits the polynomial better to the training data but also generalizes better to unseen data.")
    print("However, it's important to note that SGD comes with a trade-off, as it requires a much longer training time compared to LS.")
    

    metrics = ['Mean MSE', 'STD MSE', 'RMSE for w', 'RMSE for y(test)']
    ls_values = [mean_mse_ls.tolist(), std_mse_ls.tolist(), rmse_w_ls.tolist(), rmse_y_ls.tolist()]
    sgd_values = [mean_mse_sgd.tolist(), std_mse_sgd.tolist(), rmse_w_sgd.tolist(), rmse_y_sgd.tolist()]

    # Define the width of the bars
    bar_width = 0.35

    # Define the positions for the bars
    index = range(len(metrics))

    print("-" * 40 + "end" + "-" * 40)

    del mse_ls, std_mse_ls, mean_mse_ls
    del mse_sgd, std_mse_sgd, mean_mse_sgd

if __name__=="__main__":
    main()


In [173]:
# Define weight vector
w = torch.tensor([1, 2, 3], dtype=torch.float32).reshape(3, 1)

tensor([[1.],
        [2.],
        [3.]])

In [99]:
# Generate training set
x_train = 40.0 * (torch.rand(20, dtype=torch.float32) - 0.5).reshape(20, 1)
y_train = polynomial_fun(w, x_train)
noise_train = (0.2 * torch.randn(20, dtype=torch.float32)).reshape(20, 1)
t_train = y_train + noise_train

In [223]:
sampled_points = 40.0 * (torch.rand(20) - 0.5)
noise = 0.5 * torch.randn(20)

# Add noise to the sampled points
sampled_points_with_noise = sampled_points + noise

# Display the sampled points with noise
print(sampled_points_with_noise)




tensor([ -1.6228,  10.9914,   0.8083,  13.3597,  -4.0760,   0.8363,  18.2765,
        -11.2121,  16.5209,  -9.7214,  -9.1754,  19.6636,  -8.2949, -17.4321,
          8.8712,  -6.6705,  11.0308,   5.3499,   6.0198, -11.0572])


In [224]:
noise = 0.5 * torch.randn(20)
noise

tensor([-0.2336,  0.5072,  0.2236, -0.5370, -0.2903,  0.1069, -0.6467,  0.3978,
         0.2116, -0.2673, -0.0341, -0.6842, -0.0901, -0.5789,  0.2841, -0.0729,
        -0.1093,  0.6163, -0.4562,  0.3069])

In [ ]:
# Add Gaussian noise with standard deviation 0.5
noise = 0.5 * torch.randn(20)

# Add noise to the sampled points
sampled_points_with_noise = sampled_points + noise

# Display the sampled points with noise
print(sampled_points_with_noise)


In [89]:
# Generate testing set
x_test = 40.0 * (torch.rand(10, dtype=torch.float32) - 0.5).reshape(10, 1)
y_test = polynomial_fun(w, x_test)
noise_test = 0.2 * torch.randn(10, dtype=torch.float32).reshape(10, 1)
t_test = y_test + noise_test

In [94]:
w_hat_ls = fit_polynomial_ls(x_train, t_train, M=3)

In [220]:
fit_polynomial_sgd(x_train, t_train, 3, 0.01, 5)  # batch_size=25, learning rate=0.25

Epoch: 100 Loss 11157.46875
Epoch: 200 Loss 95.42827606201172
Epoch: 300 Loss 34.94776916503906
Epoch: 400 Loss 20.231582641601562
Epoch: 500 Loss 22.98781394958496
Epoch: 600 Loss 24.58200454711914
Epoch: 700 Loss 13.956167221069336
Epoch: 800 Loss 8.595582962036133
Epoch: 900 Loss 6.476941108703613
Epoch: 1000 Loss 4.044993877410889
Epoch: 1100 Loss 2.0603785514831543
Epoch: 1200 Loss 2.401888608932495
Epoch: 1300 Loss 0.8632366061210632
Epoch: 1400 Loss 1.4150114059448242
Epoch: 1500 Loss 0.7779735922813416
Epoch: 1600 Loss 0.5332270860671997
Epoch: 1700 Loss 0.18879815936088562
Epoch: 1800 Loss 0.17467018961906433
Epoch: 1900 Loss 0.28677135705947876
Epoch: 2000 Loss 0.11756730079650879
Epoch: 2100 Loss 0.03739427775144577
Epoch: 2200 Loss 0.03005378507077694
Epoch: 2300 Loss 0.04850706830620766
Epoch: 2400 Loss 0.06297631561756134
Epoch: 2500 Loss 0.019872542470693588
Epoch: 2600 Loss 0.2748897671699524
Epoch: 2700 Loss 144.50753784179688
Epoch: 2800 Loss 0.6763853430747986
Epoch:

Parameter containing:
tensor([[1.0101e+00, 1.9954e+00, 3.0005e+00, 4.0497e-05]], requires_grad=True)

In [180]:
min(losses)

0.04873322322964668